# CISC7021 Assignment 2: Parameter-Efficient Fine-Tuning for Mathematical Reasoning and Generalization

## Overview

This notebook adapts `Qwen/Qwen2.5-1.5B-Instruct` to grade-school math word problems
using 4-bit QLoRA supervised fine-tuning on GSM8K, then evaluates whether the
resulting improvements generalize to an external benchmark (SVAMP) or are limited to in-domain performance and output formatting. Two SFT target formats — **answer-only** and **reasoning-plus-answer** — are compared under identical prompting and decoding
conditions.

**Research questions:**
1. Does SFT on GSM8K improve only in-domain performance, or does it also improve cross-dataset generalization to SVAMP?
2. Does supervising intermediate reasoning steps help beyond supervising the final answer alone?


Sections that require you to write code are marked with `# TODO:` comments. Each one includes a short
description of what's expected.

## Required experiments (6 evaluation runs)

| Condition | Training | Supervised / generation target |
|---|---|---|
| **Original** | None (zero-shot) | Prompted to end with `#### <answer>` |
| **SFT–Answer** | 4-bit QLoRA on GSM8K | `#### <answer>` only |
| **SFT–Reasoning** | 4-bit QLoRA on the same GSM8K examples | Worked solution + `#### <answer>` |

Each of the three conditions is evaluated on **both** the GSM8K test split and the **complete** SVAMP set (evaluation-only, never used for training or model selection).
All conditions share the same chat-template prompt, greedy decoding (`do_sample=False`),
and answer-extraction logic, so differences in results reflect the training condition
rather than inconsistencies in evaluation.

## Notebook structure (maps to the assignment's Task breakdown)

- **Task 1 — Data & evaluation pipeline**: fixed GSM8K train/val split, chat-template
  construction for both SFT target formats, and the numerical answer extractor
  (handles signs, commas, decimals, simple fractions; malformed/missing answers are
  recorded as extraction failures, never manually corrected).
- **Task 2 — Original-model baseline**: zero-shot evaluation of the unmodified model
  on GSM8K test and SVAMP.
- **Task 3 — QLoRA fine-tuning**: a from-scratch, hook-based LoRA implementation
  (no `peft` library) used to train the SFT–Answer and SFT–Reasoning adapters on
  identical GSM8K examples, with training samples/epochs/hyperparameters/trainable-
  parameter counts/loss/wall-clock time/peak GPU memory logged for each run.
- **Task 4 — Evaluation & comparative analysis**: all required runs, producing
  `scores.csv` (accuracy, extraction success rate, cross-dataset generalization gap
  `Acc(GSM8K) - Acc(SVAMP)`, and accuracy change vs. the original model). You're encouraged to continue (optional) exploratory experiments.

## Metrics (as defined in the assignment)

- **Final-answer accuracy**: fraction of examples with a correct extracted answer;
  an extraction failure counts as **incorrect**, not excluded.
- **Extraction success rate**: fraction of examples with a parseable numerical
  answer after the `####` marker (format compliance).
- **Conditional accuracy**: accuracy among successfully extracted answers only —
  reported as an optional diagnostic, does not replace final-answer accuracy.
- **Cross-dataset generalization gap**: `Acc(GSM8K) - Acc(SVAMP)`; can be negative
  and is never clipped.

## Reproducibility

A fixed seed (`SEED = 42`) governs the train/val split, model initialization, and
training. All results — predictions, metrics, training logs, and the split manifest —
are saved to Google Drive so they persist across Colab sessions and satisfy the
submission requirements (`scores.csv` + six JSONL prediction files, each containing
the example identifier, raw output, extracted answer, extraction-success flag, gold
answer, correctness flag, model condition, and benchmark).



In [1]:
!pip install -q transformers datasets
!pip install -U bitsandbytes



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


^C


In [7]:
"""
Import Related Modules
"""

import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import os
from datasets import load_dataset
import json
from torch.utils.data import Dataset, DataLoader
import re
from tqdm import tqdm
import re
from decimal import Decimal, InvalidOperation
from fractions import Fraction
import torch.nn as nn
from transformers import BitsAndBytesConfig
import time
import gc
import random



In [8]:
# from google.colab import drive
import os
# drive.mount('/content/drive')
WORK_DIR = "/content/drive/MyDrive/CISC7021_Assignment2/"
CKPT_DIR = os.path.join(WORK_DIR, "ckpt")
LOG_DIR = os.path.join(WORK_DIR, "logs")
SUBMIT_DIR = os.path.join(WORK_DIR, "submission")
ANALYSIS_DIR = os.path.join(WORK_DIR, "analysis")
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(SUBMIT_DIR, exist_ok=True)
os.makedirs(ANALYSIS_DIR, exist_ok=True)

## Download Datasets

In [9]:
os.makedirs('/content/data/gsm8k', exist_ok=True)
os.makedirs('/content/data/svamp', exist_ok=True)

# ---- GSM8K ----
gsm8k = load_dataset("openai/gsm8k", "main")
gsm8k['train'].to_json('/content/data/gsm8k/train.jsonl')
gsm8k['test'].to_json('/content/data/gsm8k/test.jsonl')

# ---- SVAMP ----
svamp = load_dataset("ChilleD/SVAMP")
svamp['train'].to_json('/content/data/svamp/train.jsonl')
svamp['test'].to_json('/content/data/svamp/test.jsonl')

print("数据集已保存到 /content/data/")

Creating json from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

数据集已保存到 /content/data/


In [10]:
def read_lines(file_name):
  """
  Helper functions, returns a list of dicts
  """
  res = []
  with open(file_name, 'r') as f:
    for line in f:
      res.append(json.loads(line))
  return res



## Task 1 (part 1): Fixed Train/Validation Split

Create a fixed, reproducible train/validation split from the official GSM8K training
data using `SEED = 42`. The official GSM8K test split and the complete SVAMP set are
reserved exclusively for final evaluation and must never be used for training or
model selection.

In [11]:
gsm8k_train = read_lines('/content/data/gsm8k/train.jsonl')
gsm8k_test = read_lines('/content/data/gsm8k/test.jsonl')
svamp_train = read_lines('/content/data/svamp/train.jsonl')
svamp_test = read_lines('/content/data/svamp/test.jsonl')
from sklearn.model_selection import train_test_split
SEED = 42
val_ratio = 0.1
torch.manual_seed(SEED)
random.seed(SEED)
gsm8k_train, gsm8k_val = train_test_split(
    gsm8k_train, test_size=val_ratio, random_state=SEED
)
svamp_data = svamp_train + svamp_test
split_manifest = {
    "seed": SEED,
    "val_ratio": val_ratio,
    "gsm8k_train_size": len(gsm8k_train),
    "gsm8k_val_size": len(gsm8k_val),
    "gsm8k_test_size": len(gsm8k_test),
    "svamp_total_size": len(svamp_data),
    "gsm8k_source": "openai/gsm8k (main)",
    "svamp_source": "ChilleD/SVAMP",
    "split_method": "sklearn.model_selection.train_test_split",
}

split_manifest_path = os.path.join(LOG_DIR, "split_manifest.json")
with open(split_manifest_path, "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, indent=2, ensure_ascii=False)
print(f"saved: {split_manifest_path}")
print(f"gsm8k_train: {len(gsm8k_train)}")
print(f"gsm8k_val:   {len(gsm8k_val)}")
print(f"gsm8k_test:  {len(gsm8k_test)}")

saved: /content/drive/MyDrive/CISC7021_Assignment2/logs\split_manifest.json
gsm8k_train: 6725
gsm8k_val:   748
gsm8k_test:  1319


## Task 1 (part 2): Dataset Classes and Chat-Template Construction

`GSM8K_Dataset` and `SVAMP_Dataset` wrap raw examples into `(source, target)` pairs
using the Qwen chat template, where the user instruction states the required
`#### <answer>` format. Depending on `mode`, the target is either the final-answer
line only (`sft_answer`) or the full worked solution ending in that line
(`sft_reasoning` / `default`) — these are the two SFT target formats compared
throughout this assignment.

In [12]:

INSTRUCTION = '''Please solve the following problem. End your response with a final line in exactly this format:
#### <answer>
where <answer> is a single number (it may be negative, contain commas, be a decimal, or a simple fraction like 3/4). Do not write anything after this line.\n\n'''


class GSM8K_Dataset(Dataset):
  """
  Wraps the GSM8K dataset into (source, target) training/evaluation
  examples, formatted with the Qwen chat template.

  Depending on `mode`, the supervised target differs:
    - 'sft_answer':    target is only the final "#### <answer>" line
    - 'sft_reasoning' or 'default': target is the full worked solution ending in "#### <answer>"
  """
  def __init__(self, data, tokenizer=None, mode='default', max_len=512):
    """
    Args:
        data: a list of dicts, each with 'question' and 'answer' fields
              (raw GSM8K format).
        tokenizer: the tokenizer used to build the chat-template prompt
                   and to tokenize source/target for training.
        mode: 'sft_answer', 'sft_reasoning', or 'default'. Controls which
              target format is constructed for each example.
    """
    self.mode = mode
    self.tokenizer = tokenizer
    self.data = self.prepare_data(data)
    self.max_len = max_len

  def prepare_data(self, data):
    """
    Build (source, target) pairs for every example in `data`.

    `source` is the user prompt wrapped with the chat template, ending
    right before the assistant's turn (add_generation_prompt=True), so
    it's ready to either be fed to generate() or concatenated with a
    target for training.

    `target` is the supervised text the assistant should produce,
    format depending on self.mode.
    """
    processed_data = []
    for idx, dat in enumerate(data):
      raw_question = dat['question']
      raw_response = dat['answer']
      # if supervise on reasoning steps
      question = INSTRUCTION + raw_question
      source = self.tokenizer.apply_chat_template(
          [{"role": "user", "content": question}],
          tokenize=False,
          add_generation_prompt=True
      )
      if self.mode == 'sft_answer':
        processed_data.append(
            {
                "example_id": f"gsm8k-{idx:05d}",
                "source": source,
                "target": self.get_final_answer(raw_response)
            }
        )
      elif self.mode == 'sft_reasoning' or self.mode == 'default':
        processed_data.append(
            {
                "example_id": f"gsm8k-{idx:05d}",
                "source": source,
                "target": self.get_reasoning(raw_response)
            }
        )
      else:
        raise NotImplementedError
    return processed_data

  def get_final_answer(self, raw_answer):
    """
    Extract just the final numeric answer from the raw GSM8K 'answer'
    field and re-wrap it in the required "#### <answer>" format.

    Used for the answer-only (SFT-Answer) supervision target.
    """
    # GSM8K answers end with a line such as `#### 42`. Keep only that
    # final-answer line for answer-only supervision.
    final_answer = raw_answer.rsplit("####", 1)[-1].strip()
    return f"#### {final_answer}"
  def get_reasoning(self, raw_answer):
    """
    Extract the reasoning steps from the raw GSM8K 'answer' field.

    You're welcome to apply additional cleaning procedure.
    """
    # The raw GSM8K answer already contains the worked solution and
    # its final `#### <answer>` line.
    return raw_answer.strip()

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    """
    Tokenize one (source, target) pair and build the loss mask.

    Returns a dict with input_ids/attention_mask/labels ready to be
    batched by collate_fn. `labels` marks the source (prompt) portion
    with -100 so the loss is only computed on the target (assistant)
    portion — this is what makes it "supervised fine-tuning on the
    target" rather than training the model to also predict the prompt.
    """
    data = self.data[index]
    source = data['source']
    target = data['target']
    example_id = data['example_id']
    full_text = source + target + self.tokenizer.eos_token
    full_ids = self.tokenizer(full_text, add_special_tokens=False)["input_ids"]
    source_ids = self.tokenizer(source, add_special_tokens=False)["input_ids"]

    if len(full_ids) > self.max_len:
        dropped = len(full_ids) - self.max_len
        full_ids = full_ids[dropped:]
        supervised_start = max(0, len(source_ids) - dropped)
    else:
        supervised_start = len(source_ids)

    # TODO: build the `labels` list.
    # Requirements:
    #   - same length as full_ids
    #   - positions before `supervised_start` (the source/prompt part)
    #     must be -100, so CrossEntropyLoss ignores them
    #   - positions from `supervised_start` onward (the target/assistant
    #     part) should hold the actual token ids from full_ids, so the
    #     model is trained to predict them

    # only supervise the answer
    labels = [-100] * supervised_start + full_ids[supervised_start:]


    return {
        "source": source,
        "example_id": example_id,
        "target": target,
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


def collate_fn(batch):
  """
  Collate a list of __getitem__ outputs into a single padded batch.

  Uses left-padding. `input_ids` is padded with pad_token_id and
  `attention_mask` with 0, but `labels` must be padded with -100
  (NOT any real token id) so that padding positions are also excluded
  from the loss — for the same reason the source/prompt portion was
  masked with -100 in __getitem__.
  """
  source = [item['source'] for item in batch]
  target = [item['target'] for item in batch]
  input_ids = [item['input_ids'] for item in batch]
  attention_mask = [item['attention_mask'] for item in batch]
  labels = [item['labels'] for item in batch]
  example_ids = [item['example_id'] for item in batch]
  max_len = max(len(ids) for ids in input_ids)
  # TODO: left-pad input_ids, attention_mask, and labels to max_len.
  # Requirements:
  #   - input_ids: pad with tokenizer.pad_token_id
  #   - attention_mask: pad with 0 (padding positions should not be attended to)
  #   - labels: pad with -100 (padding positions should not contribute to the loss)
  #   - all padding goes on the LEFT (prepended), matching the rest of
  #     this codebase's left-padding convention


  input_ids = [([tokenizer.pad_token_id] * (max_len - len(ids))) + ids for ids in input_ids]
  attention_mask = [([0] * (max_len - len(mask))) + mask for mask in attention_mask]
  labels = [([-100] * (max_len - len(l))) + l for l in labels]

  return {
      "example_id": example_ids,
      "source": source,
      "target": target,
      "input_ids": torch.tensor(input_ids, dtype=torch.long),
      "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
      "labels": torch.tensor(labels, dtype=torch.long),
  }


class SVAMP_Dataset(Dataset):
  def __init__(self, data, tokenizer=None):
    self.tokenizer = tokenizer
    self.data = self.prepare_data(data)
    self.max_len = 1024

  def prepare_data(self, data):
    processed_data = []
    for dat in data:
      body = str(dat['Body']).strip()
      question_text = str(dat['Question']).strip()
      raw_answer = dat['Answer']
      example_id = dat['ID']

      question = INSTRUCTION + f"{body} {question_text}".strip()
      source = self.tokenizer.apply_chat_template(
          [{"role": "user", "content": question}],
          tokenize=False,
          add_generation_prompt=True
      )
      processed_data.append(
          {
              "ID": example_id,
              "source": source,
              "target": self.get_final_answer(raw_answer)
          }
      )
    return processed_data

  def get_final_answer(self, raw_answer):
    return f"#### {raw_answer}"

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    # returns a <source, target> pair
    data = self.data[index]
    source = data['source']
    target = data['target']
    full_text = source + target + self.tokenizer.eos_token
    full_ids = self.tokenizer(full_text, add_special_tokens=False)["input_ids"]
    source_ids = self.tokenizer(source, add_special_tokens=False)["input_ids"]

    assert full_ids[:len(source_ids)] == source_ids

    if len(full_ids) > self.max_len:
        dropped = len(full_ids) - self.max_len
        full_ids = full_ids[dropped:]
        supervised_start = max(0, len(source_ids) - dropped)
    else:
        supervised_start = len(source_ids)

    labels = [-100] * supervised_start + full_ids[supervised_start:]
    # raise NotImplementedError("Build the labels list with the -100 mask")


    return {
        "source": source,
        "target": target,
        "example_id": data['ID'],
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

## Setup Model & Dataset

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
lm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", attn_implementation="sdpa", device_map="auto", quantization_config=quant_config)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
lm.to("cuda")
print(lm, lm.device, lm.dtype)


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
# =========================================================
# Prepare all dataset
# =========================================================

gsm8k_train_dataset_answer_only = GSM8K_Dataset(gsm8k_train, tokenizer, mode='sft_answer')
gsm8k_val_dataset_answer_only = GSM8K_Dataset(gsm8k_val, tokenizer, mode='sft_answer')

gsm8k_train_dataset_reasoning = GSM8K_Dataset(gsm8k_train, tokenizer, mode='sft_reasoning')
gsm8k_val_dataset_reasoning = GSM8K_Dataset(gsm8k_val, tokenizer, mode='sft_reasoning')

gsm8k_test_dataset_default = GSM8K_Dataset(gsm8k_test, tokenizer, mode='default')

svamp_dataset = SVAMP_Dataset(svamp_data, tokenizer)

In [ ]:
# Dataset format checks: these do not train or generate anything.
# They verify the contracts used by the later training/evaluation cells.
_sample = gsm8k_train_dataset_answer_only[0]
assert {"source", "target", "input_ids", "attention_mask", "labels"}.issubset(_sample)
assert len(_sample["input_ids"]) == len(_sample["attention_mask"]) == len(_sample["labels"])
assert _sample["target"].lstrip().startswith("####")
_target_ids = tokenizer(_sample["target"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
assert _sample["labels"][-len(_target_ids):] == _target_ids
assert -100 in _sample["labels"]

_batch = collate_fn([gsm8k_train_dataset_answer_only[0], gsm8k_train_dataset_answer_only[1]])
assert _batch["input_ids"].ndim == 2
assert _batch["input_ids"].shape == _batch["attention_mask"].shape == _batch["labels"].shape
assert _batch["input_ids"].shape[0] == 2
assert torch.all((_batch["attention_mask"] == 0) | (_batch["attention_mask"] == 1))
assert torch.all(_batch["labels"][_batch["attention_mask"] == 0] == -100)
print("Dataset format checks passed.")


## Task 1 (part 3): Answer Extraction and Normalization

A deterministic numerical answer extractor that reads the mandatory final
`#### <answer>` line, handling optional signs, commas, decimals, and simple
fractions. A malformed or missing answer is recorded as an extraction failure — never
manually corrected — since this directly affects final-answer accuracy (Eq. 1).

In [ ]:

NUMBER_PATTERN = r"[+-]?(?:(?:\d[\d,]*)(?:\.\d*)?|\.\d+)(?:\s*/\s*[+-]?(?:(?:\d[\d,]*)(?:\.\d*)?|\.\d+))?"
FINAL_ANSWER_RE = re.compile(rf"^\s*####\s*({NUMBER_PATTERN})\s*$")
MARKER_RE = re.compile(r"^\s*####")

def normalize_number(text):
    """
    Convert a string-form number into an exact Fraction, so it can be
    compared reliably against the gold answer. Returns None on failure.

    Must handle, at minimum:
      - optional leading sign (+/-)
      - thousands separators (commas), e.g. "1,200"
      - decimals, e.g. "3.14", ".5"
      - simple fractions, e.g. "3/4", with each side possibly signed/decimal
      - division by zero, or any unparseable input -> return None
        (never raise an exception out of this function)
    """
    if text is None:
        return None
    candidate = str(text).strip()
    if not re.fullmatch(NUMBER_PATTERN, candidate):
        return None

    # TODO: parse `candidate` into a Fraction.
    # Hints:
    #   - Fraction(Decimal(x)) gives an exact conversion from a decimal
    #     string to a Fraction (avoids floating-point rounding issues)
    #   - split on "/" (at most once) if a fraction is present
    #   - remember to strip out commas before converting
    #   - wrap the conversion in try/except and return None on
    #     InvalidOperation / ValueError / ZeroDivisionError
    # Solution:
    
    try:
        if "/" in candidate:
            numerator_text, denominator_text = re.split(r"\s*/\s*", candidate, maxsplit=1)
            numerator = Fraction(Decimal(numerator_text.replace(",", "")))
            denominator = Fraction(Decimal(denominator_text.replace(",", "")))
            if denominator == 0:
                return None
            return numerator / denominator
        return Fraction(Decimal(candidate.replace(",", "")))
    except (InvalidOperation, ValueError, ZeroDivisionError):
        return None
    # raise NotImplementedError("Parse candidate into a Fraction")


def extract_final_answer(generation):
    """
    Extract the answer from the '#### <answer>' line in a model's
    generated text.

    Rule: there must be exactly one line starting with '####', and it
    must be the last line; otherwise this counts as an extraction failure.

    Returns: (raw_answer_string, normalized_Fraction) on success,
    or (None, None) on failure.
    """
    if not isinstance(generation, str):
        return None, None

    lines = generation.rstrip().splitlines()
    marker_lines = [line for line in lines if MARKER_RE.match(line)]

    # TODO: implement the extraction logic described above.
    # Requirements:
    #   - len(lines) == 0                       -> failure, return (None, None)
    #   - number of marker_lines != 1            -> failure (missing OR duplicated marker)
    #   - the marker line is not the last line, or doesn't fully match
    #     FINAL_ANSWER_RE                        -> failure
    #   - otherwise, extract the raw answer text (FINAL_ANSWER_RE's
    #     capture group) and normalize it with normalize_number; if
    #     normalization fails, also treat as failure

    if len(lines) == 0 or len(marker_lines) != 1:
        return None, None
    match = FINAL_ANSWER_RE.fullmatch(lines[-1])
    if match is None:
        return None, None
    raw_answer = match.group(1)
    normalized = normalize_number(raw_answer)
    return (raw_answer, normalized) if normalized is not None else (None, None)
    # raise NotImplementedError("Implement the extraction logic")


def fraction_to_string(value):
    """Convert a Fraction back into a readable string; integers are shown without a denominator."""
    if value.denominator == 1:
        return str(value.numerator)
    return f"{value.numerator}/{value.denominator}"


def get_eos_ids(model, tokenizer):
    eos_ids = model.generation_config.eos_token_id
    if eos_ids is None:
        eos_ids = tokenizer.eos_token_id
    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]
    else:
        eos_ids = list(eos_ids)

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is not None and im_end_id != tokenizer.unk_token_id and im_end_id not in eos_ids:
        eos_ids.append(im_end_id)
    return eos_ids


In [ ]:
# Quick checks: run this cell before any GPU training.
# Passing means the deterministic parts of the assignment are ready.
assert normalize_number("1,200") == Fraction(1200)
assert normalize_number("-3.5") == Fraction(-7, 2)
assert normalize_number("3/4") == Fraction(3, 4)
assert normalize_number("1/0") is None
assert extract_final_answer("Work\n#### 42") == ("42", Fraction(42))
assert extract_final_answer("#### 42\nextra") == (None, None)
assert extract_final_answer("#### 42\n#### 43") == (None, None)
print("Parser checks passed. Run the metrics check after the evaluation helper cell.")


## Task 2 Evaluation Pipeline
`evaluate()` runs greedy, deterministic generation (`do_sample=False`) over a dataset,
extracts each answer, and compares it against the gold answer to compute accuracy,
extraction success rate, and conditional accuracy. This same function is reused
across all three conditions (Original / SFT–Answer / SFT–Reasoning) and both
benchmarks (GSM8K, SVAMP), so the prompt template and decoding settings are
guaranteed to stay identical across the required comparison.

In [ ]:
def evaluate(model, tokenizer, dataset, condition, benchmark, max_new_tokens,batch_size=1, device="cuda"):
  """
  Run greedy generation over `dataset`, extract the final answer from
  each output, and compare it against the gold answer.

  Args:
      model: the model to evaluate (with or without LoRA hooks attached).
      tokenizer: tokenizer used for encoding prompts and decoding outputs.
      dataset: a Dataset whose __getitem__ returns source/target/example_id.
      condition: label for this run, e.g. "original", "sft_answer",
                 "sft_reasoning" — recorded in every prediction record.
      benchmark: label for the dataset being evaluated, e.g. "gsm8k",
                 "svamp" — recorded in every prediction record.
      max_new_tokens: generation length limit.
      batch_size: number of examples per batch.
      device: device to run generation on.

  Returns:
      (predictions, metrics):
          predictions: a list of per-example result dicts.
          metrics: a dict of aggregate scores (see compute_metrics).
  """
  model.eval()
  tokenizer.padding_side = "left"

  eos_ids = get_eos_ids(model, tokenizer)
  loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
  # ---- Stats ----
  torch.cuda.reset_peak_memory_stats(device)
  eval_start_time = time.perf_counter()

  predictions = []
  for batch in tqdm(loader, desc="evaluating"):
    sources = batch['source']
    targets = batch['target']
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    example_ids = batch['example_id']

    # Encode only the prompts. The chat template was already applied
    # when the dataset was constructed, so no extra special tokens
    # should be added here.
    encoded = tokenizer(
        sources,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )


    input_len = encoded["input_ids"].shape[1]
    generated_ids = output_ids[:, input_len:]
    raw_outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    for example_id, source, target, raw_output in zip(example_ids, sources, targets, raw_outputs):
        extracted_text, extracted_value = extract_final_answer(raw_output)
        _, gold_value = extract_final_answer(target)

        correct = (
            extracted_value is not None
            and gold_value is not None
            and extracted_value == gold_value
        )

        predictions.append({
            "example_id": example_id,
            "source": source,
            "raw_output": raw_output,
            "extracted_answer": extracted_text,
            "extraction_success": extracted_value is not None,
            "gold_answer": target,
            "correct": bool(correct),
            "condition": condition,
            "benchmark": benchmark,
        })

      # ---- Collect Stats ----
  eval_elapsed_seconds = time.perf_counter() - eval_start_time
  peak_memory_gb = torch.cuda.max_memory_allocated(device) / 1024**3

  print(f"Total evaluation wall-clock time: {eval_elapsed_seconds:.1f} sec "
        f"({eval_elapsed_seconds/60:.2f} min)")
  print(f"Peak GPU memory during evaluation: {peak_memory_gb:.2f} GB")

  metrics = compute_metrics(predictions)
  metrics["wall_clock_seconds"] = eval_elapsed_seconds
  metrics["peak_gpu_memory_gb"] = peak_memory_gb
  metrics['condition'] = condition
  metrics['benchmark'] = benchmark

  return predictions, metrics

def compute_metrics(predictions):
    """
    Aggregate per-example predictions into summary metrics.

    Each prediction has "extraction_success" (bool) and "correct" (bool).
    Per the assignment's metric definitions, an extraction failure must
    be counted as an INCORRECT answer for the purposes of accuracy — it
    is not simply excluded.

    Returns a dict with the following required fields:
        - n_examples: total number of examples evaluated
        - accuracy: fraction of all examples answered correctly
                    (extraction failures count as wrong)
        - extraction_success_rate: fraction of examples where a
                    parseable numerical answer was produced at all
        - conditional_accuracy: accuracy computed ONLY among examples
                    where extraction succeeded; required for all six
                    evaluation runs, alongside accuracy and extraction
                    success rate; does not replace accuracy.
                    Return 0.0 if nothing was extracted.
        - n_extracted: number of examples with a successful extraction
        - n_correct: number of examples with a correct answer
    """
    n = len(predictions)

    # TODO: compute n_extracted, n_correct, accuracy, extraction_success_rate,
    # and conditional_accuracy from `predictions`, following the definitions
    # in the docstring above.
    n_extracted = sum(p["extraction_success"] for p in predictions)
    n_correct = sum(p["correct"] for p in predictions)
    # raise NotImplementedError("Compute accuracy / extraction_success_rate / conditional_accuracy")

    return {
        "n_examples": n,
        "accuracy": accuracy,
        "extraction_success_rate": extraction_success_rate,
        "conditional_accuracy": conditional_accuracy,
        "n_extracted": n_extracted,
        "n_correct": n_correct,
    }


In [ ]:
# =========================================================
# Task 2: Original-Model Baseline Evaluation (2 benchmarks)
# =========================================================
import gc
import json
import pandas as pd

all_predictions = {}
all_metrics = []

def save_predictions_jsonl(predictions, condition, benchmark, out_dir):
    path = os.path.join(out_dir, f"{condition}_{benchmark}_predictions.jsonl")
    with open(path, "w", encoding="utf-8") as f:
        for record in predictions:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"saved: {path}")
    return path

def save_metrics_json(metrics, condition, benchmark, out_dir):
    path = os.path.join(out_dir, f"{condition}_{benchmark}_metrics.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"saved: {path}")
    return path

def load_predictions_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def load_metrics_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def run_and_save(model, tokenizer, ds, condition, benchmark, **kwargs):
    pred_path = os.path.join(SUBMIT_DIR, f"{condition}_{benchmark}_predictions.jsonl")
    metrics_path = os.path.join(SUBMIT_DIR, f"{condition}_{benchmark}_metrics.json")

    if os.path.exists(pred_path) and os.path.exists(metrics_path):
        print(f"skip (already exists): {pred_path}")
        preds = load_predictions_jsonl(pred_path)
        metrics = load_metrics_json(metrics_path)
    else:
        preds, metrics = evaluate(model, tokenizer, ds, condition=condition, benchmark=benchmark, **kwargs)
        save_predictions_jsonl(preds, condition, benchmark, SUBMIT_DIR)
        save_metrics_json(metrics, condition, benchmark, SUBMIT_DIR)

    all_predictions[(condition, benchmark)] = preds
    all_metrics.append(metrics)
    return preds, metrics

benchmarks = [("gsm8k", gsm8k_test_dataset_default), ("svamp", svamp_dataset)]

# ---- Original (zero-shot, no LoRA) ----
for benchmark_name, ds in benchmarks:
    run_and_save(lm, tokenizer, ds, condition="original", benchmark=benchmark_name,
                 max_new_tokens=256, batch_size=8, device="cuda")

## Task 3. Implementing a LoRA Training Pipeline

Low Rank Adaptation (LoRA) is a parameter-efficient fine-tuning method that freezes the pretrained weights of a large model and injects small trainable low-rank matrices into each layer to approximate weight updates. This drastically reduces the number of trainable parameters and memory needed for fine-tuning while achieving performance comparable to full fine-tuning on many tasks.

In the following section, we will implement a LoRA SFT pipeline from scratch: define LoRA modules, inject and detach them via PyTorch Hooks, and train them with the supervision dataset we built.

In [ ]:
class LoRAHook:
    """
    Implements LoRA (Low-Rank Adaptation) as a forward hook.

    Instead of modifying the original nn.Linear layer's structure, this
    registers a forward hook that adds a low-rank branch (B @ A) on top
    of the layer's output after it has been computed:

        output = base_layer(x) + scaling * (x @ A) @ B

    A and B are two trainable matrices, much smaller than the original
    weight matrix (hence "low-rank"). The base_layer itself stays frozen
    and is never updated during training.
    """
    def __init__(self, module, rank=8, alpha=16, dropout=0.1):
        """
        Initialize a LoRA hook to be attached to a given nn.Linear layer.

        Args:
            module: The original layer to inject LoRA into (typically
                     nn.Linear or a quantized Linear4bit layer). This
                     function does not freeze its parameters; freezing is
                     handled by the external training code.
            rank: Rank r of the low-rank matrices. Smaller rank means
                  fewer trainable parameters.
            alpha: Numerator of the scaling factor; the final scaling
                   applied is alpha / rank.
            dropout: Dropout probability applied to the input of the
                     LoRA branch.
        """
        super().__init__()
        self.base_layer = module
        self.rank = rank
        self.alpha = alpha
        self.in_features = self.base_layer.in_features
        self.out_features = self.base_layer.out_features
        self.dropout = nn.Dropout(dropout)
        device = next(module.parameters()).device
        self.handle = None
        # TODO: Initialize lora_A and lora_B as trainable parameters.
        # Requirements:
        #   1. lora_A has shape (in_features, rank), lora_B has shape (rank, out_features)
        #   2. Initialize them so that at the start of training, the LoRA
        #      branch's output is exactly 0 (i.e. it doesn't change the
        #      original model's behavior yet), while still being able to
        #      receive gradients and be updated during training.
        #   3. Think about it: why can't both matrices be initialized to
        #      all zeros? Why can't both be initialized randomly?
        # self.lora_A = ...
        # self.lora_B = ...
        # Initialization
        # Solution:
        # self.lora_A = nn.Parameter(torch.zeros(self.in_features, rank, device=device))
        # self.lora_B = nn.Parameter(torch.zeros(rank, self.out_features, device=device))
        # nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
        # nn.init.zeros_(self.lora_B)
        # self.scaling = alpha / rank


    def __call__(self, module, input, output):
        """
        The forward hook callback, automatically invoked right after
        base_layer.forward() has been executed.

        Args:
            module: The layer that triggered this hook (i.e. base_layer itself).
            input: The original input passed to base_layer.forward, as a
                   tuple;
            output: The raw output of base_layer.forward(x), before any
                    LoRA contribution is added.

        Returns:
            The new output with the LoRA low-rank branch added on top;
            this automatically replaces the original output.
        """
        x = input[0]
        # TODO: Compute the LoRA branch's output and add it to the
        # original output, then return it.
        # Hints:
        #   - Pass x through dropout first, then multiply by lora_A, then
        #     lora_B, then multiply by scaling.
        # Solution:
        # lora_x = self.dropout(x).to(self.lora_A.dtype)
        # lora_out = (lora_x @ self.lora_A) @ self.lora_B * self.scaling
        # return output + lora_out.to(output.dtype)
        raise NotImplementedError("Compute the LoRA branch output and add it to output")



    def register(self):
        self.handle = self.base_layer.register_forward_hook(self)
        return self

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None


def inject_lora(model, target_modules=None, rank=8, alpha=16, dropout=0.1):
    """
    Walk through all submodules of the model and attach a LoRAHook to
    every nn.Linear layer whose name matches target_modules.

    Args:
        model: The model to inject LoRA into.
        target_modules: A list of layer names to target (e.g.
                          ["q_proj", "v_proj"]). Pass None to attach LoRA
                          to every nn.Linear layer in the model.
        rank: Rank of the low-rank matrices, passed to each LoRAHook.
        alpha: Scaling numerator, passed to each LoRAHook.
        dropout: Dropout probability for the LoRA branch, passed to each LoRAHook.

    Returns:
        A dict mapping each layer's full path name to its LoRAHook instance.
    """
    lora_hooks = {}
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        leaf_name = name.split(".")[-1]
        if target_modules is not None and leaf_name not in target_modules:
            continue
        hook = LoRAHook(module, rank=rank, alpha=alpha, dropout=dropout).register()
        lora_hooks[name] = hook
    return lora_hooks

def detach_lora(lora_hooks):
    """detach all LoRA hooks, reset model's behavior"""
    for hook in lora_hooks.values():
        hook.remove()

def save_lora_adapter(lora_hooks, path, rank, alpha, dropout, target_modules):
    """
    Save only lora_A/lora_B and the corresponding hyperparameter config,
    without touching the base model's weights.
    lora_hooks: the dict {layer_name: LoRAHook instance} returned by inject_lora
    """
    state = {
        "config": {
            "rank": rank,
            "alpha": alpha,
            "dropout": dropout,
            "target_modules": target_modules,
        },
        "weights": {
            name: {
                "lora_A": hook.lora_A.detach().cpu(),
                "lora_B": hook.lora_B.detach().cpu(),
            }
            for name, hook in lora_hooks.items()
        },
    }
    torch.save(state, path)

def freeze_and_collect_trainable_params(model, lora_hooks):
    for p in model.parameters():
        p.requires_grad = False

    trainable_params = []
    for hook in lora_hooks.values():
        hook.lora_A.requires_grad = True
        hook.lora_B.requires_grad = True
        trainable_params.extend([hook.lora_A, hook.lora_B])
    return trainable_params

def load_lora_weights(lora_hooks, state, device="cuda"):
    """
    state: the full dict returned by torch.load(path) (contains "config" and "weights")
    lora_hooks: hooks that have already been injected, with a structure matching `state`
    """
    for name, hook in lora_hooks.items():
        assert name in state["weights"], f"权重文件里缺少这一层: {name}"
        hook.lora_A.data.copy_(state["weights"][name]["lora_A"].to(device))
        hook.lora_B.data.copy_(state["weights"][name]["lora_B"].to(device))


def load_lora_adapter(model, path, device="cuda"):
    """
    Starting from scratch: inject the LoRA structure into a clean model,
    then load the trained weights into it.
    """
    state = torch.load(path, map_location=device)
    config = state["config"]

    lora_hooks = inject_lora(
        model,
        target_modules=config["target_modules"],
        rank=config["rank"],
        alpha=config["alpha"],
        dropout=config["dropout"],
    )
    load_lora_weights(lora_hooks, state, device=device)
    return lora_hooks

def infer_lora_config(lora_hooks):
    """
    Infer the LoRA hyperparameters (rank, alpha, dropout, target module
    names) directly from an already-injected set of LoRAHooks, instead
    of requiring the caller to pass them in separately (and risk them
    getting out of sync with the actual hooks).
    """
    if not lora_hooks:
        raise ValueError("lora_hooks is empty, cannot infer config")

    sample_hook = next(iter(lora_hooks.values()))
    return {
        "rank": sample_hook.rank,
        "alpha": sample_hook.alpha,
        "dropout": sample_hook.dropout.p,
        "target_modules": sorted({name.split(".")[-1] for name in lora_hooks.keys()}),
    }

In [ ]:
def check_lora_hook_init(rank=4, alpha=8, in_features=16, out_features=16):
    """
    Sanity check for LoRAHook.__init__: verifies shapes, that lora_A is
    not all-zero, that the LoRA branch's contribution is exactly zero
    right after initialization, and that the hooked layer's actual
    output matches the original (un-hooked) layer's output.
    """
    layer = nn.Linear(in_features, out_features)
    x = torch.randn(2, in_features)
    original_output = layer(x)   # capture the output BEFORE the hook is registered

    hook = LoRAHook(layer, rank=rank, alpha=alpha)

    assert hook.lora_A.shape == (in_features, rank), \
        f"lora_A should have shape {(in_features, rank)}, got {tuple(hook.lora_A.shape)}"
    assert hook.lora_B.shape == (rank, out_features), \
        f"lora_B should have shape {(rank, out_features)}, got {tuple(hook.lora_B.shape)}"
    assert not torch.allclose(hook.lora_A, torch.zeros_like(hook.lora_A)), \
        "lora_A should NOT be all-zero (otherwise gradients can never reach lora_B)"
    assert torch.allclose(hook.lora_A @ hook.lora_B, torch.zeros_like(hook.lora_A @ hook.lora_B)), \
        "lora_A @ lora_B should be exactly zero right after initialization"

    hook.register()
    hooked_output = layer(x)   # now the hook is active; this should still match original_output

    assert torch.allclose(hooked_output, original_output, atol=1e-5), \
        "Right after initialization, the hooked layer's output should be identical to the original layer's output"

    hook.remove()
    print("LoRAHook sanity check passed")


check_lora_hook_init()

## Task 3: Training LoRA Modules
Train two LoRA adapters — SFT–Answer and SFT–Reasoning — on the same fixed GSM8K
training examples, using 4-bit QLoRA with the required hyperparameters:

*Required hyperparameters (per assignment spec):**
- Base model: `Qwen/Qwen2.5-1.5B-Instruct`, loaded in 4-bit NF4 with double quantization, FP16/BF16 compute dtype
- Number of epochs: 2
- Maximum sequence length: 512
- Learning rate: 1e-5
- Optimizer: AdamW (no LR scheduler unless you add one)
- LoRA rank: 8, alpha: 16, dropout: 0.1
- Target modules: all `nn.Linear` layers (attention + MLP projections + LM Head, you're welcome to specify your surgical plan)
- Train/validation split: fixed 90/10 split of the official GSM8K training set (`SEED=42`)

We train **two separate LoRA adapters** on the same GSM8K training examples, differing only in the supervised target format:
- **SFT–Answer**: supervises only the final `#### <answer>` line
- **SFT–Reasoning**: supervises the full worked solution followed by `#### <answer>`

Everything else (prompt template, decoding settings, evaluation protocol) is kept identical across both conditions.

For each run we record: number of training/validation examples, number of trainable parameters (and % of total), training/validation loss per epoch, wall-clock training time, and peak GPU memory usage — all saved to `LOG_DIR` for inclusion in the final report.


In [ ]:


def train_lora(model, tokenizer, lora_hooks, train_dataset, val_dataset, num_epochs=2, batch_size=8, learning_rate=1e-4, device="cuda", max_len=512):
    """
    Train a LoRA adapter on top of a frozen base model.

    Reports the training/validation loss history, along with wall-clock
    time and peak GPU memory usage. You are welcome to modify the
    training loop to improve the performance of your LoRA model.

    Args:
        model: the base model, already with LoRA hooks attached via
               inject_lora (passed in as `lora_hooks`).
        tokenizer: tokenizer used by collate_fn's padding logic.
        lora_hooks: dict {layer_name: LoRAHook} returned by inject_lora.
        train_dataset, val_dataset: Datasets whose __getitem__ returns
               tokenized (input_ids, attention_mask, labels) examples.
        num_epochs, batch_size, learning_rate: standard training
               hyperparameters.
        device: device to train on.

    Returns:
        history: a dict with "train_loss", "val_loss" (one entry per
                 epoch), plus "wall_clock_seconds" and "peak_gpu_memory_gb".
    """


    # Freeze the quantized base model and optimize only the LoRA
    # matrices attached to the selected layers.
    for param in model.parameters():
        param.requires_grad = False

    trainable_params = []
    for hook in lora_hooks.values():
        hook.lora_A.requires_grad = True
        hook.lora_B.requires_grad = True
        trainable_params.extend([hook.lora_A, hook.lora_B])

    # The total count refers to the base model. LoRA parameters live
    # on hook objects and are counted separately below.
    n_total_params = sum(p.numel() for p in model.parameters())
    n_trainable_params = sum(p.numel() for p in trainable_params)

    print(f"trainable params: {sum(p.numel() for p in trainable_params):,} "
          f"/ total: {sum(p.numel() for p in model.parameters()):,}") # You should see ~10M v.s. total: ~0.8B

    # ---- Stats reset ----
    torch.cuda.reset_peak_memory_stats(device)
    train_start_time = time.perf_counter()
    history = {"train_loss": [], "val_loss": []}
    # Training Loop
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate)
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{num_epochs}"):
            # remember that sources are the questions and targets are the expected answers, so we'll have to concatenate them to form the input for the model. The model will be trained to predict the targets given the sources.
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # TODO: implement one training step.
            # Requirements:
            #   1. Run the forward pass with input_ids, attention_mask, and labels.
            #   2. Extract the scalar loss from the model output.
            #   3. Backpropagate with loss.backward().
            #   4. Update parameters, then clear gradients.
            #   5. Add loss.item() to total_train_loss.
            # Solution:
            # outputs = model(
            #     input_ids=input_ids,
            #     attention_mask=attention_mask,
            #     labels=labels,
            # )
            # loss = outputs.loss
            # loss.backward()
            # optimizer.step()
            # optimizer.zero_grad()
            # total_train_loss += loss.item()
            raise NotImplementedError("Implement forward -> backward -> optimizer step")


        train_loss = total_train_loss / len(train_loader)
        history["train_loss"].append(train_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {train_loss:.4f}")
        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}/{num_epochs}"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

                val_loss += outputs.loss.item()
                # break
        val_loss /= len(val_loader)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {val_loss:.4f}")

     # ---- Collect stats ----
    train_elapsed_seconds = time.perf_counter() - train_start_time
    peak_memory_gb = torch.cuda.max_memory_allocated(device) / 1024**3

    print(f"Total training wall-clock time: {train_elapsed_seconds:.1f} sec "
          f"({train_elapsed_seconds/60:.2f} min)")
    print(f"Peak GPU memory during training: {peak_memory_gb:.2f} GB")

    history["wall_clock_seconds"] = train_elapsed_seconds
    history["peak_gpu_memory_gb"] = peak_memory_gb
    history["n_train_examples"] = len(train_dataset)
    history["n_val_examples"] = len(val_dataset)
    history["seed"] = SEED
    history["num_epochs"] = num_epochs

    history["max_seq_length"] = max_len
    history["learning_rate"] = learning_rate
    history["effective_batch_size"] = batch_size
    history["optimizer"] = "AdamW"
    history["scheduler"] = "none"

    lora_config = infer_lora_config(lora_hooks)
    history["lora_rank"] = lora_config["rank"]
    history["lora_alpha"] = lora_config["alpha"]
    history["lora_dropout"] = lora_config["dropout"]
    history["lora_target_modules"] = lora_config["target_modules"]


    history["n_trainable_params"] = n_trainable_params
    history["n_total_params"] = n_total_params
    history["trainable_param_pct"] = n_trainable_params / n_total_params
    return history



In [ ]:
# =========================================================
# Required Hyperparameters
# =========================================================
NUM_EPOCHS = 2
MAX_SEQ_LENGTH = 512
LEARNING_RATE = 1e-5

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = None  # None = all nn.Linear layers

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 8

MAX_NEW_TOKENS = 256

In [ ]:
# =========================================================
# SFT-Answer
# =========================================================
lora_hooks = inject_lora(lm, target_modules=LORA_TARGET_MODULES, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)

if not os.path.exists(CKPT_DIR + "/sft_answer_lora.pt"):
    history_answer = train_lora(
        lm, tokenizer, lora_hooks=lora_hooks,
        train_dataset=gsm8k_train_dataset_answer_only, val_dataset=gsm8k_val_dataset_answer_only,
        num_epochs=NUM_EPOCHS, batch_size=TRAIN_BATCH_SIZE, learning_rate=LEARNING_RATE, device="cuda",
    )
    save_lora_adapter(lora_hooks, CKPT_DIR + "/sft_answer_lora.pt", rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES)
    with open(os.path.join(LOG_DIR, "sft_answer_train_history.json"), "w") as f:
        json.dump(history_answer, f, indent=2)
    print(history_answer)

detach_lora(lora_hooks)
del lora_hooks
gc.collect()
torch.cuda.empty_cache()

trainable params: 10,460,160 / total: 888,616,448


Training Epoch 1/2: 100%|██████████| 3363/3363 [56:09<00:00,  1.00s/it]


Epoch 1/2, Training Loss: 0.7125


Validation Epoch 1/2: 100%|██████████| 374/374 [03:09<00:00,  1.97it/s]


Epoch 1/2, Validation Loss: 0.6584


Training Epoch 2/2: 100%|██████████| 3363/3363 [56:00<00:00,  1.00it/s]


Epoch 2/2, Training Loss: 0.5583


Validation Epoch 2/2: 100%|██████████| 374/374 [03:08<00:00,  1.98it/s]


Epoch 2/2, Validation Loss: 0.6811
Total training wall-clock time: 7108.2 sec (118.47 min)
Peak GPU memory during training: 6.74 GB
{'train_loss': [0.7125230922335728, 0.5582835678493179], 'val_loss': [0.6583679940471834, 0.6811268071439814], 'wall_clock_seconds': 7108.225229495, 'peak_gpu_memory_gb': 6.737638473510742}


In [ ]:
# =========================================================
# SFT-Reasoning
# =========================================================
lora_hooks = inject_lora(lm, target_modules=LORA_TARGET_MODULES, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)

if not os.path.exists(CKPT_DIR + "/sft_reasoning_lora.pt"):
    history_reasoning = train_lora(
        lm, tokenizer, lora_hooks=lora_hooks,
        train_dataset=gsm8k_train_dataset_reasoning, val_dataset=gsm8k_val_dataset_reasoning,
        num_epochs=NUM_EPOCHS, batch_size=TRAIN_BATCH_SIZE, learning_rate=LEARNING_RATE, device="cuda",
    )
    save_lora_adapter(lora_hooks, CKPT_DIR + "/sft_reasoning_lora.pt", rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES)
    with open(os.path.join(LOG_DIR, "sft_reasoning_train_history.json"), "w") as f:
        json.dump(history_reasoning, f, indent=2)
    print(history_reasoning)

detach_lora(lora_hooks)
del lora_hooks
gc.collect()
torch.cuda.empty_cache()

trainable params: 10,460,160 / total: 888,616,448


Training Epoch 1/2: 100%|██████████| 3363/3363 [1:35:19<00:00,  1.70s/it]


Epoch 1/2, Training Loss: 0.3970


Validation Epoch 1/2: 100%|██████████| 374/374 [05:14<00:00,  1.19it/s]


Epoch 1/2, Validation Loss: 0.3770


Training Epoch 2/2: 100%|██████████| 3363/3363 [1:35:18<00:00,  1.70s/it]


Epoch 2/2, Training Loss: 0.3237


Validation Epoch 2/2: 100%|██████████| 374/374 [05:15<00:00,  1.19it/s]


Epoch 2/2, Validation Loss: 0.3745
Total training wall-clock time: 12068.5 sec (201.14 min)
Peak GPU memory during training: 10.04 GB
{'train_loss': [0.39696824664503133, 0.32372310682976135], 'val_loss': [0.37702203085795444, 0.3745041324652453], 'wall_clock_seconds': 12068.456640025, 'peak_gpu_memory_gb': 10.035137176513672}


## Task 4: Required LoRA Evaluation Runs

Evaluate all LoRA conditions (SFT–Answer / SFT–Reasoning) on both
benchmarks (GSM8K test, complete SVAMP) — four required runs in total. Each run's
predictions and metrics are written to disk immediately after evaluation completes,
so partial progress survives a Colab disconnect and already-completed runs are
skipped on rerun.

In [ ]:
# =========================================================
# Run 4 LoRA evaluations
# =========================================================

# Just in case you restarted the kernal:
if "all_predictions" not in dir():
    all_predictions = {}
if "all_metrics" not in dir():
    all_metrics = []
if "benchmarks" not in dir():
    benchmarks = [("gsm8k", gsm8k_test_dataset_default), ("svamp", svamp_dataset)]

# ---- SFT-Answer ----
lora_hooks = load_lora_adapter(lm, CKPT_DIR + "/sft_answer_lora.pt", device="cuda")
for benchmark_name, ds in benchmarks:
    run_and_save(lm, tokenizer, ds, condition="sft_answer", benchmark=benchmark_name,
                 max_new_tokens=MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE, device="cuda")
detach_lora(lora_hooks)
del lora_hooks
gc.collect()
torch.cuda.empty_cache()

# ---- SFT-Reasoning ----
lora_hooks = load_lora_adapter(lm, CKPT_DIR + "/sft_reasoning_lora.pt", device="cuda")
for benchmark_name, ds in benchmarks:
    run_and_save(lm, tokenizer, ds, condition="sft_reasoning", benchmark=benchmark_name,
                 max_new_tokens=MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE, device="cuda")
detach_lora(lora_hooks)
del lora_hooks
gc.collect()
torch.cuda.empty_cache()


skip (already exists): /content/drive/MyDrive/CISC7021_Assignment2/submission/original_gsm8k_predictions.jsonl
skip (already exists): /content/drive/MyDrive/CISC7021_Assignment2/submission/original_svamp_predictions.jsonl
skip (already exists): /content/drive/MyDrive/CISC7021_Assignment2/submission/sft_answer_gsm8k_predictions.jsonl
skip (already exists): /content/drive/MyDrive/CISC7021_Assignment2/submission/sft_answer_svamp_predictions.jsonl
skip (already exists): /content/drive/MyDrive/CISC7021_Assignment2/submission/sft_reasoning_gsm8k_predictions.jsonl
skip (already exists): /content/drive/MyDrive/CISC7021_Assignment2/submission/sft_reasoning_svamp_predictions.jsonl


## Task 4: Main Results Table

Assemble `scores.csv` from the six saved metrics files: final-answer accuracy,
extraction success rate, conditional accuracy, the cross-dataset generalization gap
(`Acc(GSM8K) - Acc(SVAMP)`), and each fine-tuned condition's accuracy change relative
to the Original baseline — the evidence needed to answer this assignment's research
questions.

In [ ]:
# =========================================================
# Prepare score.csv
# =========================================================
import os
import json
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/CISC7021_Assignment2/"
SUBMIT_DIR = os.path.join(WORK_DIR, "submission")

conditions = ["original", "sft_answer", "sft_reasoning"]
benchmark_names = ["gsm8k", "svamp"]

all_metrics = []
for condition in conditions:
    for benchmark_name in benchmark_names:
        metrics_path = os.path.join(SUBMIT_DIR, f"{condition}_{benchmark_name}_metrics.json")
        pred_path = os.path.join(SUBMIT_DIR, f"{condition}_{benchmark_name}_predictions.jsonl")

        assert os.path.exists(metrics_path), f"metrics missing: {metrics_path}"
        assert os.path.exists(pred_path), f"predictions missing: {pred_path}"

        with open(metrics_path, encoding="utf-8") as f:
            metrics = json.load(f)
        all_metrics.append(metrics)

assert len(all_metrics) == 6, f"6 records required，but have {len(all_metrics)}"

scores = pd.DataFrame(all_metrics)

accuracy_lookup = scores.set_index(["condition", "benchmark"])["accuracy"].to_dict()
gaps = {c: accuracy_lookup[(c, "gsm8k")] - accuracy_lookup[(c, "svamp")] for c in conditions}
baseline_accuracy = {b: accuracy_lookup[("original", b)] for b in benchmark_names}

scores["generalization_gap_svamp"] = scores["condition"].map(gaps)
scores["accuracy_change_vs_original"] = scores.apply(
    lambda row: row["accuracy"] - baseline_accuracy[row["benchmark"]], axis=1
)

scores = scores[[
    "condition", "benchmark", "n_examples", "accuracy", "extraction_success_rate",
    "conditional_accuracy", "generalization_gap_svamp", "accuracy_change_vs_original",
    "n_extracted", "n_correct", "wall_clock_seconds", "peak_gpu_memory_gb",
]].sort_values(["condition", "benchmark"])

scores_path = os.path.join(SUBMIT_DIR, "scores.csv")
scores.to_csv(scores_path, index=False)
display(scores)
print(f"saved: {scores_path}")

,condition,benchmark,n_examples,accuracy,extraction_success_rate,conditional_accuracy,generalization_gap_svamp,accuracy_change_vs_original,n_extracted,n_correct,wall_clock_seconds,peak_gpu_memory_gb
0,original,gsm8k,1319,0.000000,0.006065,0.000000,0.000000,0.000000,8,0,4584.008994,1.426450
1,original,svamp,1000,0.000000,0.005000,0.000000,0.000000,0.000000,5,0,3024.186300,1.390869
2,sft_answer,gsm8k,1319,0.117513,1.000000,0.117513,-0.209487,0.117513,1319,155,391.729261,1.589809
3,sft_answer,svamp,1000,0.327000,1.000000,0.327000,-0.209487,0.327000,1000,327,255.499167,1.443841
4,sft_reasoning,gsm8k,1319,0.459439,0.975739,0.470862,-0.054561,0.459439,1287,606,5220.344930,1.519668
5,sft_reasoning,svamp,1000,0.514000,0.998000,0.515030,-0.054561,0.514000,998,514,2094.987882,1.372985


saved: /content/drive/MyDrive/CISC7021_Assignment2/submission/scores.csv


In [ ]:
# =========================================================
# sanity check
# =========================================================
required_prediction_files = sorted(f for f in os.listdir(SUBMIT_DIR) if f.endswith("_predictions.jsonl"))
assert len(required_prediction_files) == 6, f"6 predictions required, but have {len(required_prediction_files)}"

required_fields = {"example_id", "raw_output", "extracted_answer", "extraction_success",
                    "gold_answer", "correct", "condition", "benchmark"}

for fname in required_prediction_files:
    path = os.path.join(SUBMIT_DIR, fname)
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    assert rows, f"file empty: {fname}"
    assert required_fields.issubset(rows[0].keys()), f"{fname} field missing: {required_fields - rows[0].keys()}"

print("Sanity check passed")

## Task 4 (continued): Results Analysis & Exploration

Starting from this section, you are expected to move from *running* experiments to
*interpreting* them. Use `scores.csv` and the six prediction/metrics files produced
above as your evidence base, and refer back to the assignment handout (Task 4,
"Evaluation and Comparative Analysis" and "Questions to Address in the Report") for
exactly what your analysis and error analysis need to cover.

You are welcome to reuse and adapt any of the functions defined earlier in this
notebook (`evaluate`, `compute_metrics`, etc.) for further exploration in this
section.

Anything explored here beyond the six required runs should be clearly labeled as
exploratory and kept separate from the required results — per the assignment,
optional findings cannot substitute for a missing required run.


## A Note on Optional Extensions

If you complete an optional extension (Section "Optional Extensions" in the
assignment handout), save **all** of its outputs under a separate `optional/`
directory — never inside `submission/`, and never overwriting any required
result file:

```python
OPTIONAL_DIR = os.path.join(WORK_DIR, "optional")
os.makedirs(OPTIONAL_DIR, exist_ok=True)
```

Use a clear subfolder and naming structure to distinguish different experiments/configurations.

In your report, when you refer to an optional result, add a **footnote** pointing
to the corresponding path, e.g.:

> We additionally explored a higher LoRA rank (r=16).[^1]
>
> [^1]: Results saved under `optional/rank_16/` in the submitted notebook/results archive.

This keeps the required six-run comparison in the main table unambiguous, while
still letting a reader trace any optional finding back to its exact output files.

In [ ]:
"""
Please Add Your Analysis Code Here!
"""

## Submission checklist and report evidence

After the full run finishes, this notebook provides all required artifacts:

- `submission/scores.csv`: six model-benchmark rows with accuracy, extraction success rate, generalization gap, and changes from baseline.
- `submission/*_predictions.jsonl` and `submission/*_metrics.json`: prediction and metrics files for all six required runs (one prediction + one metrics file per model-benchmark combination).
- `logs/sft_answer_train_history.json` and `logs/sft_reasoning_train_history.json`: training settings, losses, elapsed time, and peak GPU memory.
- `logs/split_manifest.json`: the fixed GSM8K train/val/test split and SVAMP source provenance, for reproducibility.


For the 2-5 page report, use the displayed main table and evidence from the paired cases to answer whether GSM8K fine-tuning improves SVAMP performance, whether changes come from accuracy or formatting, and whether reasoning supervision changes accuracy or failure modes. Do not invent outcomes before running the six evaluations.



### Use of Generative AI

I acknowledge the use of Claude to assist with notebook structure, implementation, and debugging. I reviewed, tested, and revised the generated material and take responsibility for the submitted work. No external model was used to generate or alter the required benchmark predictions.


## Reference Runtime and Results

The following figures are reference values from one Colab T4 run. They are provided to help identify major runtime or output-format problems; small differences are expected across runs and students are not required to reproduce every value exactly.

### Approximate Runtime

| Stage | Original | SFT-Answer | SFT-Reasoning |
| --- | ---: | ---: | ---: |
| Train | - | ~2 h | ~2 h |
| GSM8K evaluation | ~1 h | ~3 min | ~1 h |
| SVAMP evaluation | ~25 min | ~2 min | ~25 min |

### Reference Resource Usage

- Colab GPU: NVIDIA T4
- System RAM: 6.66 / 12.67 GB
- GPU RAM: 4.20 / 15.00 GB
- Disk: 58.30 / 112.64 GB

### Reference Results

| Model | Dataset | Accuracy |
| --- | --- | ---: |
| Original | GSM8K | 2.50% |
| Original | SVAMP | 0.70% |
| SFT-Answer | GSM8K | 9.70% |
| SFT-Answer | SVAMP | 38.70% |
| SFT-Reasoning | GSM8K | 45.34% |
| SFT-Reasoning | SVAMP | 57.70% |

Use your own saved predictions and metrics as the authoritative results for the report. These reference values are not a substitute for running the required experiments.
